# TabTune on SageMaker Processing

Use a CPU-only Python 3.11 Studio kernel. The notebook plans and schedules jobs; model checkpoints download inside isolated GPU Processing jobs. Read the cloud section of `README.md` and complete the image build and GPU smoke gates before a full run.


## 1. Install and authenticate

In a fresh Studio environment, install `pip install -e ".[cloud]"` from the repository root. Configure an authenticated `sa-east-1` AWS identity, a private S3 prefix, a Processing role, and a digest-pinned ECR image. The caller needs S3, SageMaker, ECR, IAM, Service Quotas and CloudWatch access. The job role also needs checkpoint-host access. See README for details.


In [ ]:
import pandas as pd
from tabtune_baseline import (BenchmarkConfig, SageMakerConfig,
    plan_sagemaker_baseline, run_sagemaker_baseline, resume_sagemaker_baseline)

df = pd.read_parquet("input.parquet")  # local Studio data; edit path and column names
benchmark = BenchmarkConfig(id_col="id", target_col="target", split_col="split")
cloud = SageMakerConfig(
    role_arn="arn:aws:iam::123456789012:role/your-processing-role",
    image_uri="123456789012.dkr.ecr.sa-east-1.amazonaws.com/your-repo@sha256:" + "0" * 64,
    staging_uri="s3://your-private-bucket/tabtune-runs/",
)


## 2. Review the complete plan

This cell reads the installed TabTune registry and the public AWS price list. It does not upload input, create a job, or download a checkpoint. Replace all example AWS identifiers above. Costs are on-demand estimates in USD with no run spending cap; the regional fallback snapshot is dated in each candidate. The resource formula is uncalibrated until the real GPU smoke.


In [ ]:
plan = plan_sagemaker_baseline(df, benchmark, cloud)
rows = []
for model, remote in zip(plan.baseline.models, plan.models):
    first = remote.candidates[0] if remote.candidates else None
    rows.append({"model": model.name, "status": model.status,
                 "reason": model.reason or "; ".join(remote.unavailable_reasons),
                 "context_rows": model.context_rows, "features": len(model.feature_cols),
                 "estimated_ram_gib": remote.estimate.get("cpu_ram_gib"),
                 "estimated_vram_per_gpu_gib": remote.estimate.get("vram_per_gpu_gib"),
                 "cheapest_type": first.instance_type if first else None,
                 "estimated_cost_usd": first.estimated_cost_usd if first else None,
                 "price_source": first.price_source if first else None,
                 "price_at": first.price_at if first else None})
display(pd.DataFrame(rows))
print("plan identity:", plan.identity)
print("minimum first-attempt estimate USD:", sum(r["estimated_cost_usd"] or 0 for r in rows))


## 3. Launch after review

The run stages one checksum-verified snapshot in your private S3 prefix, starts at most three GPU jobs, and prints structured status every 60 seconds. Keep the run URI. A failed notebook can be resumed. Staged input is retained until a terminal run and successful local artifact copy.


In [ ]:
result = run_sagemaker_baseline(df, plan, "./baseline-runs")
print(result.run_uri, result.status)
if result.local_result:
    print(result.local_result.report_html)
    display(pd.DataFrame(result.leaderboard()))


## 4. Resume or retrieve

Run this cell in a new Studio session with the URI printed above. An active lease makes the second notebook read-only. Completed runs copy report, manifest, JSONL events and row-level scores into Studio. Check `results.json` for quota, checkpoint, IAM, timeout, OOM, invalid input and unknown failures.


In [ ]:
run_uri = "s3://your-private-bucket/tabtune-runs/<run-id>/"  # paste actual URI
resumed = resume_sagemaker_baseline(run_uri, "./baseline-runs")
print(resumed.status, resumed.model_statuses)
if resumed.local_result:
    print(resumed.local_result.report_html)
